# 03 – Administrative Units

**Objective:** Download, inspect, and process the official Gdynia district boundaries from the
Gdynia Open Data Portal, then save a clean GeoPackage to
`data/processed/admin_units/gdynia_districts.gpkg`.

See [docs/admin_units.md](../docs/admin_units.md) for dataset details.

In [ ]:
from __future__ import annotations

import io
import zipfile

import geopandas as gpd
import requests
import yaml
from pathlib import Path

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
PROJECT_ROOT  = Path.cwd().parent
CONFIG_PATH   = PROJECT_ROOT / "config" / "config.yaml"

with CONFIG_PATH.open() as fh:
    cfg = yaml.safe_load(fh)

DISTRICTS_URL  = cfg["sources"]["districts"]["url"]
LOCAL_ZIP      = PROJECT_ROOT / cfg["sources"]["districts"]["local_path"]
DISTRICTS_GPKG = PROJECT_ROOT / cfg["database"]["districts_path"]
PROJECT_CRS    = cfg["crs"]["project_crs"]

LOCAL_ZIP.parent.mkdir(parents=True, exist_ok=True)
DISTRICTS_GPKG.parent.mkdir(parents=True, exist_ok=True)

print("Districts URL:", DISTRICTS_URL)

## 1. Download district ZIP

In [ ]:
if not LOCAL_ZIP.exists():
    print("Downloading Dzielnice_Gdyni.zip …")
    resp = requests.get(DISTRICTS_URL, timeout=60)
    resp.raise_for_status()
    LOCAL_ZIP.write_bytes(resp.content)
    print(f"  Saved {len(resp.content):,} bytes")
else:
    print(f"Using cached: {LOCAL_ZIP}")

## 2. Inspect ZIP contents

In [ ]:
with zipfile.ZipFile(LOCAL_ZIP) as zf:
    names = zf.namelist()
    print("Files in archive:")
    for n in names:
        print(" ", n)

## 3. Read and explore

In [ ]:
gdf = gpd.read_file(f"zip://{LOCAL_ZIP}")
print(f"Rows: {len(gdf)}  |  CRS: {gdf.crs}")
print("Columns:", list(gdf.columns))
gdf.head()

In [ ]:
# Visualise district boundaries
ax = gdf.plot(figsize=(8, 8), edgecolor="black", facecolor="lightblue", alpha=0.5)
ax.set_title("Gdynia Districts")
ax.set_axis_off()

## 4. Reproject and save

In [ ]:
if str(gdf.crs) != PROJECT_CRS:
    gdf = gdf.to_crs(PROJECT_CRS)
    print(f"Reprojected to {PROJECT_CRS}")

gdf.to_file(DISTRICTS_GPKG, driver="GPKG", layer="gdynia_districts")
print(f"Saved {len(gdf)} districts to {DISTRICTS_GPKG}")